# Assisted (exo) vs unassisted (Awinda) — same exo TCN checkpoints

Paper section *Estimation During Assisted and Unassisted Walking*:

- **Assisted**: existing batch metrics from `compare_processed_hip_exo_id.ipynb` (logged hip-exo model) and `compare_processed_knee_exo_id.ipynb` (knee encoder replay, no-LPF pipeline).
- **Unassisted**: Awinda IMU IK angles for the **target joint only** replayed offline through the **same checkpoints** used on the exoskeletons.

| Joint | Shared conditions | Exo model checkpoint | Exo evaluation source |
|-------|-------------------|----------------------|------------------------|
| Hip R | LG + RA | `runs/0512_ik_id_hip_offline_zero_phase/best_model.pt` | logged telemetry (`compare_processed_hip_exo_id_metrics.csv`) |
| Knee R | RA + RD | `runs/0707_knee_finetune_balanced_lg_ra_rd/best_model.pt` | encoder replay (`compare_processed_knee_exo_id_replay_metrics.csv`) |

**Hip replay (Awinda)**: deploy-matched causal input LPF (6 Hz angle, 15 Hz velocity) → TCN → zero-phase 6 Hz output LPF (matches hip-exo eval on logged `model_out_nmpkg_raw`).

**Knee replay (Awinda)**: no input LPF — raw IMU IK angle + B-spline velocity → TCN → zero-phase 6 Hz output LPF (matches knee encoder replay).

**GT**: unassisted OpenSim ID / body mass (right side). Post-sync **10 s** trim at each end (same as exo compare notebooks). Awinda trials are time-aligned via Awinda↔Vicon IK angle xcorr (5 s skip), as in `process_awinda.ipynb`.

Outputs: per-trial CSV, condition summaries, and LaTeX-ready overall means for the paper bracket placeholders.

In [ ]:
import inspect
import io
import json
import pickle
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.interpolate import splrep, splev
from scipy.signal import butter, sosfilt, sosfiltfilt

warnings.filterwarnings('ignore', message='.*NumPy.*')

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()

def _resolve_processed_root() -> Path:
    candidates = [
        Path('/media/metamobility3/Samsung_T52/Results/processed'),
        Path('/media/metamobility3/Samsung_T53/Results/processed'),
        Path('/media/metamobility3/Samsung_T5/Results/processed'),
    ]
    for root in candidates:
        if (root / 'AB01_Jinwoo' / 'awinda' / 'id').is_dir():
            return root
    raise FileNotFoundError('No processed root with awinda/id found; mount Samsung drive.')

PROCESSED_ROOT = _resolve_processed_root()
IMU_IK_ROOT = Path('/home/metamobility3/Jinwoo/mt_processed')
IMU_IK_METHOD = 'VQF'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CACHE_DIR = PROJECT_ROOT / 'analysis' / 'cache'
OUT_DIR = PROJECT_ROOT / 'analysis' / 'paper_outputs' / 'assisted_unassisted'
OUT_DIR.mkdir(parents=True, exist_ok=True)

HIP_EXO_METRICS_CSV = CACHE_DIR / 'compare_processed_hip_exo_id_metrics.csv'
KNEE_EXO_METRICS_CSV = CACHE_DIR / 'compare_processed_knee_exo_id_replay_metrics.csv'

HIP_CKPT = PROJECT_ROOT / 'runs/0512_ik_id_hip_offline_zero_phase/best_model.pt'
KNEE_CKPT = PROJECT_ROOT / 'runs/0707_knee_finetune_balanced_lg_ra_rd/best_model.pt'

JOINT = 'hip_flexion_r'
KNEE_JOINT = 'knee_angle_r'
MOMENT_COL_HIP = 'hip_flexion_r_moment'
MOMENT_COL_KNEE = 'knee_angle_r_moment'

HIP_SHARED_TASKS = {'LG', 'RA'}
KNEE_SHARED_TASKS = {'RA', 'RD'}

LPF_CUTOFF_HZ, LPF_ORDER = 6.0, 4
TRIM_START_SEC = 10.0
TRIM_END_SEC = 10.0
XCORR_SKIP_SEC = 5.0
ALIGN_MAX_LAG_SEC = 30.0
KNEE_MODEL_OUT_ALIGN_LAG_SAMPLES = -6

# Hip deploy LPF (hip-exo-ctrl-V2/cfg/final.yaml)
HIP_ANGLE_LPF_HZ, HIP_ANGLE_LPF_ORDER = 6.0, 4
HIP_VEL_LPF_HZ, HIP_VEL_LPF_ORDER = 15.0, 4

SUBJECT_MASS_KG = {
    'AB01_Jinwoo': 88.0, 'AB02_Oscar': 71.1, 'AB03_Ilseung': 84.4,
    'AB04_Changseob': 74.0, 'AB05_Maria': 55.0, 'AB06_Jimin': 82.6,
    'AB07_Amy': 51.3, 'AB08_Seokhyun': 71.9,
}

ANGLE_OFFSET_DEG = {
    ('AB02_Oscar', 'LG_0p8mps'): {'hip_flexion_r': 6.0, 'knee_angle_r': 6.0},
    ('AB01_Jinwoo', 'RD_0p8mps'): {'knee_angle_r': 10.0},
}

ALL_CHANNELS = [
    'hip_flexion_r', 'knee_angle_r', 'ankle_angle_r',
    'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l',
]
XCORR_CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in (
    'hip_flexion_r', 'knee_angle_r', 'hip_flexion_l', 'knee_angle_l',
)]

sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES
from model import TCN

ALL_IK_CHANNEL_IDX = [IK_DOF_NAMES.index(c) for c in ALL_CHANNELS]

print(f'Device: {DEVICE}')
print(f'Hip exo metrics: {HIP_EXO_METRICS_CSV} (exists={HIP_EXO_METRICS_CSV.is_file()})')
print(f'Knee exo metrics: {KNEE_EXO_METRICS_CSV} (exists={KNEE_EXO_METRICS_CSV.is_file()})')
print(f'Processed root: {PROCESSED_ROOT}')
print(f'Hip checkpoint: {HIP_CKPT} (exists={HIP_CKPT.is_file()})')
print(f'Knee checkpoint: {KNEE_CKPT} (exists={KNEE_CKPT.is_file()})')

In [ ]:
def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(arr) < 4:
        return arr.copy()
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, arr) if mode == 'zero_phase' else sosfilt(sos, arr)


def lpf_nan(x, fs_hz, cutoff_hz, order, mode):
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(3, order + 1):
        return arr
    if finite.all():
        return butter_lpf(arr, fs_hz, cutoff_hz, order, mode)
    idx = np.arange(arr.size, dtype=np.float64)
    filled = np.interp(idx, idx[finite], arr[finite])
    out = butter_lpf(filled, fs_hz, cutoff_hz, order, mode)
    out[~finite] = np.nan
    return out


def lpf_mc(X, fs_hz, cutoff_hz, order, mode='zero_phase'):
    return np.column_stack([
        lpf_nan(X[:, c], fs_hz, cutoff_hz, order, mode) for c in range(X.shape[1])
    ])


def rmse_r2(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else float('nan')
    return rmse, r2


def analysis_trim_mask(t, trim_start_s=TRIM_START_SEC, trim_end_s=TRIM_END_SEC):
    t_rel = np.asarray(t, dtype=np.float64) - np.nanmin(t)
    t_end = float(np.nanmax(t_rel))
    return (t_rel >= float(trim_start_s)) & (t_rel <= t_end - float(trim_end_s))


def parse_opensim_table(path: Path) -> pd.DataFrame:
    with open(path) as f:
        header_end = next(i for i, line in enumerate(f) if line.strip().lower() == 'endheader')
    return pd.read_csv(path, sep=r'\s+', skiprows=header_end + 1).set_index('time')


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def filename_to_condition(stem: str) -> str:
    speed, cond = stem.split('_', 1)
    return f'{cond.upper()}_{speed}'


def build_model_input_from_pkl(imu_dict: dict) -> np.ndarray:
    n = len(next(iter(imu_dict.values())))
    pos_deg = np.zeros((n, len(IK_DOF_NAMES)), dtype=np.float64)
    key_map = {
        'hip_flexion_r': 'hip_flexion_r', 'knee_angle_r': 'knee_flexion_r', 'ankle_angle_r': 'ankle_flexion_r',
        'hip_flexion_l': 'hip_flexion_l', 'knee_angle_l': 'knee_flexion_l', 'ankle_angle_l': 'ankle_flexion_l',
    }
    sign_map = {'knee_angle_r': -1.0, 'knee_angle_l': -1.0}
    for ik_name, pkl_name in key_map.items():
        idx = IK_DOF_NAMES.index(ik_name)
        pos_deg[:, idx] = sign_map.get(ik_name, 1.0) * np.asarray(imu_dict[pkl_name], dtype=np.float64)
    return pos_deg


def apply_angle_offset_rad(pos_rad, subject, condition):
    offsets = ANGLE_OFFSET_DEG.get((subject, condition))
    if not offsets:
        return pos_rad
    out = np.asarray(pos_rad, dtype=np.float64).copy()
    for name, deg in offsets.items():
        out[:, IK_DOF_NAMES.index(name)] += np.deg2rad(float(deg))
    return out


def build_vicon_ik_rad(ik_df: pd.DataFrame) -> np.ndarray:
    pos_deg = np.full((len(ik_df), len(IK_DOF_NAMES)), np.nan)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(dtype=np.float64)
    return np.deg2rad(pos_deg)


def _zscore_1d(x):
    x = np.asarray(x, dtype=np.float64)
    m = np.isfinite(x)
    if m.sum() < 2:
        return np.zeros_like(x)
    mu, sd = float(np.nanmean(x[m])), float(np.nanstd(x[m]))
    if sd < 1e-9:
        return np.zeros_like(x)
    out = (x - mu) / sd
    out[~m] = 0.0
    return out


def estimate_lag_from_angle_xcorr(awinda_rad, vicon_rad, fs_hz, max_lag_samples,
                                  channel_idx=None, skip_s=XCORR_SKIP_SEC,
                                  angle_cutoff=6.0, filter_order=4, in_mode='zero_phase'):
    channel_idx = list(channel_idx or XCORR_CHANNEL_IDX)
    awinda_f = lpf_mc(awinda_rad[:, ALL_IK_CHANNEL_IDX], fs_hz, angle_cutoff, filter_order, in_mode)
    vicon_f = lpf_mc(vicon_rad[:, ALL_IK_CHANNEL_IDX], fs_hz, angle_cutoff, filter_order, in_mode)
    skip = int(round(skip_s * fs_hz))
    best_lag, best_score = 0, -np.inf
    for lag in range(-max_lag_samples, max_lag_samples + 1):
        start_a, start_b = max(lag, 0), max(-lag, 0)
        nn = min(len(awinda_f) - start_a, len(vicon_f) - start_b) - skip
        if nn < int(round(2.0 * fs_hz)):
            continue
        seg_a = awinda_f[start_a + skip:start_a + skip + nn]
        seg_b = vicon_f[start_b + skip:start_b + skip + nn]
        score = sum(float(np.dot(_zscore_1d(seg_a[:, c]), _zscore_1d(seg_b[:, c])) / nn)
                    for c in channel_idx)
        if score > best_score:
            best_score, best_lag = score, lag
    lag = int(np.clip(best_lag, -max_lag_samples, max_lag_samples))
    return lag, float(best_score)


class _CausalLowPass:
    def __init__(self, fs_hz, cutoff_hz, order=4):
        self.order = max(1, int(order))
        if cutoff_hz <= 0:
            self.alpha = 1.0
        else:
            dt = 1.0 / float(fs_hz)
            tau = 1.0 / (2.0 * np.pi * float(cutoff_hz))
            self.alpha = dt / (tau + dt)
        self.state = [0.0] * self.order
        self.initialized = False

    def update(self, x):
        x = float(x)
        if not self.initialized:
            self.state = [x] * self.order
            self.initialized = True
            return x
        y = x
        for i in range(self.order):
            self.state[i] = self.state[i] + self.alpha * (y - self.state[i])
            y = self.state[i]
        return float(y)


def apply_causal_lpf_series(x, fs_hz, cutoff_hz, order):
    lpf = _CausalLowPass(fs_hz, cutoff_hz, order)
    return np.asarray([lpf.update(float(v)) for v in np.asarray(x, dtype=np.float64)], dtype=np.float64)


def causal_backward_derivative(x, fs_hz):
    arr = np.asarray(x, dtype=np.float64)
    dt = 1.0 / float(fs_hz)
    vel = np.zeros_like(arr)
    if len(arr) > 1:
        vel[1:] = (arr[1:] - arr[:-1]) / dt
    return vel


def _tcn_ctor_kwargs(cfg):
    allowed = {k for k in inspect.signature(TCN.__init__).parameters if k != 'self'}
    return {k: v for k, v in cfg.items() if k in allowed}


def load_tcn(checkpoint: Path):
    ckpt = torch.load(str(checkpoint), map_location=DEVICE, weights_only=False)
    cfg = ckpt['model_config']
    model = TCN(**_tcn_ctor_kwargs(cfg))
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    window_size = int(ckpt.get('window_size', 100))
    return model, window_size, cfg


@torch.no_grad()
def run_unilateral_tcn(model, angle, vel, window_size):
    angle = np.asarray(angle, dtype=np.float32)
    vel = np.asarray(vel, dtype=np.float32)
    n = int(min(len(angle), len(vel)))
    pred = np.zeros(n, dtype=np.float32)
    model.eval()
    for t in range(n):
        start = max(0, t - window_size + 1)
        valid = t - start + 1
        x = np.zeros((2, window_size), dtype=np.float32)
        x[0, -valid:] = angle[start:t + 1]
        x[1, -valid:] = vel[start:t + 1]
        xt = torch.from_numpy(x).unsqueeze(0).to(device=DEVICE, dtype=torch.float32)
        pred[t] = float(model(xt)[0, 0, -1].item())
    return pred.astype(np.float64)


def _shift_samples_1d(x, lag):
    x = np.asarray(x, dtype=np.float64)
    out = np.full_like(x, np.nan)
    lag = int(lag)
    if lag == 0:
        out[:] = x
    elif lag > 0:
        out[lag:] = x[:len(x) - lag]
    else:
        out[:lag] = x[-lag:]
    return out


def list_awinda_trials():
    rows = []
    for subj_dir in sorted(IMU_IK_ROOT.glob('AB*')):
        if not subj_dir.is_dir():
            continue
        subject = subj_dir.name
        ik_dir = subj_dir / 'ik' / IMU_IK_METHOD
        if not ik_dir.exists():
            ik_dir = subj_dir
        for pkl_path in sorted(ik_dir.glob('*.pkl')):
            cond = filename_to_condition(pkl_path.stem)
            id_path = PROCESSED_ROOT / subject / 'awinda' / 'id' / f'{cond}_id.sto'
            vicon_path = PROCESSED_ROOT / subject / 'awinda' / 'ik' / f'{cond}_ik.mot'
            if not (id_path.is_file() and vicon_path.is_file()):
                continue
            task = cond.split('_', 1)[0]
            rows.append({
                'subject': subject,
                'condition': cond,
                'task': task,
                'trial_key': f'{subject}::{cond}',
                'pkl_path': pkl_path,
                'id_path': id_path,
                'vicon_path': vicon_path,
                'mass_kg': SUBJECT_MASS_KG[subject],
            })
    return pd.DataFrame(rows)


print('Helpers ready.')

In [ ]:
def load_exo_assisted_metrics() -> Tuple[pd.DataFrame, pd.DataFrame]:
    hip = pd.read_csv(HIP_EXO_METRICS_CSV)
    hip = hip[hip['task'].isin(HIP_SHARED_TASKS)].copy()
    hip['platform'] = 'hip_exo_assisted'
    hip['channel'] = JOINT

    knee = pd.read_csv(KNEE_EXO_METRICS_CSV)
    knee = knee[knee['task'].isin(KNEE_SHARED_TASKS)].copy()
    knee = knee.rename(columns={
        'rmse_encoder_replay_nmpkg': 'rmse_nmpkg',
        'r2_encoder_replay_nmpkg': 'r2_nmpkg',
    })
    knee['platform'] = 'knee_exo_assisted'
    knee['channel'] = KNEE_JOINT
    return hip, knee


def process_awinda_hip_trial(row, hip_model, hip_window) -> Dict:
    subject, cond = row['subject'], row['condition']
    imu = pickle.load(open(row['pkl_path'], 'rb'))
    pos_rad = apply_angle_offset_rad(np.deg2rad(build_model_input_from_pkl(imu)), subject, cond)
    vicon_rad = build_vicon_ik_rad(parse_opensim_table(row['vicon_path']))

    id_df = parse_opensim_table(row['id_path'])
    t_id = id_df.index.to_numpy(dtype=np.float64)
    fs_hz = 1.0 / float(np.median(np.diff(t_id))) if len(t_id) > 2 else 100.0

    ch_idx = IK_DOF_NAMES.index(JOINT)
    angle_raw = pos_rad[:, ch_idx]
    angle_causal = apply_causal_lpf_series(angle_raw, fs_hz, HIP_ANGLE_LPF_HZ, HIP_ANGLE_LPF_ORDER)
    vel_raw = causal_backward_derivative(angle_raw, fs_hz)
    vel_causal = apply_causal_lpf_series(vel_raw, fs_hz, HIP_VEL_LPF_HZ, HIP_VEL_LPF_ORDER)

    pred_raw = run_unilateral_tcn(hip_model, angle_causal, vel_causal, hip_window)
    pred_nmpkg = lpf_nan(pred_raw, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, 'zero_phase')

    id_nm = id_df[MOMENT_COL_HIP].to_numpy(dtype=np.float64)
    gt_raw = id_nm / row['mass_kg']
    gt_nmpkg_full = lpf_nan(gt_raw, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, 'causal')

    max_lag = int(round(ALIGN_MAX_LAG_SEC * fs_hz))
    lag, xcorr_score = estimate_lag_from_angle_xcorr(pos_rad, vicon_rad, fs_hz, max_lag)

    start_pred, start_id = max(lag, 0), max(-lag, 0)
    n_sync = min(len(pred_nmpkg) - start_pred, len(gt_nmpkg_full) - start_id)
    pred_sync = pred_nmpkg[start_pred:start_pred + n_sync]
    gt_sync = gt_nmpkg_full[start_id:start_id + n_sync]
    t_sync = t_id[start_id:start_id + n_sync]

    trim = analysis_trim_mask(t_sync)
    rmse, r2 = rmse_r2(gt_sync[trim], pred_sync[trim])
    return {
        'trial_key': row['trial_key'], 'subject': subject, 'task': row['task'],
        'condition': cond, 'platform': 'awinda_hip_only_exo_model',
        'channel': JOINT, 'rmse_nmpkg': rmse, 'r2_nmpkg': r2,
        'lag_samples': lag, 'xcorr_score': xcorr_score, 'n_samples': int(trim.sum()),
    }


def _vicon_ik_velocity_spline(t, angle_rad):
    t = np.asarray(t, dtype=np.float64)
    angle_rad = np.asarray(angle_rad, dtype=np.float64)
    if len(t) < 5:
        fs = 1.0 / float(np.median(np.diff(t))) if len(t) > 2 else 100.0
        return causal_backward_derivative(angle_rad, fs)
    tck = splrep(t, angle_rad, s=0, k=3)
    return np.asarray(splev(t, tck, der=1), dtype=np.float64)


def process_awinda_knee_trial(row, knee_model, knee_window) -> Dict:
    subject, cond = row['subject'], row['condition']
    imu = pickle.load(open(row['pkl_path'], 'rb'))
    pos_rad = apply_angle_offset_rad(np.deg2rad(build_model_input_from_pkl(imu)), subject, cond)
    vicon_rad = build_vicon_ik_rad(parse_opensim_table(row['vicon_path']))

    id_df = parse_opensim_table(row['id_path'])
    t_id = id_df.index.to_numpy(dtype=np.float64)
    fs_hz = 1.0 / float(np.median(np.diff(t_id))) if len(t_id) > 2 else 100.0

    ch_idx = IK_DOF_NAMES.index(KNEE_JOINT)
    angle_raw = pos_rad[:, ch_idx]
    vel_raw = _vicon_ik_velocity_spline(np.arange(len(angle_raw)) / fs_hz, angle_raw)

    pred_raw = run_unilateral_tcn(knee_model, angle_raw, vel_raw, knee_window)
    pred_raw = _shift_samples_1d(pred_raw, KNEE_MODEL_OUT_ALIGN_LAG_SAMPLES)
    pred_nmpkg = lpf_nan(pred_raw, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, 'zero_phase')

    id_nm = id_df[MOMENT_COL_KNEE].to_numpy(dtype=np.float64)
    gt_nmpkg_full = lpf_nan(id_nm / row['mass_kg'], fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, 'zero_phase')

    max_lag = int(round(ALIGN_MAX_LAG_SEC * fs_hz))
    lag, xcorr_score = estimate_lag_from_angle_xcorr(pos_rad, vicon_rad, fs_hz, max_lag)

    start_pred, start_id = max(lag, 0), max(-lag, 0)
    n_sync = min(len(pred_nmpkg) - start_pred, len(gt_nmpkg_full) - start_id)
    pred_sync = pred_nmpkg[start_pred:start_pred + n_sync]
    gt_sync = gt_nmpkg_full[start_id:start_id + n_sync]
    t_sync = t_id[start_id:start_id + n_sync]

    trim = analysis_trim_mask(t_sync)
    rmse, r2 = rmse_r2(gt_sync[trim], pred_sync[trim])
    return {
        'trial_key': row['trial_key'], 'subject': subject, 'task': row['task'],
        'condition': cond, 'platform': 'awinda_knee_only_exo_model',
        'channel': KNEE_JOINT, 'rmse_nmpkg': rmse, 'r2_nmpkg': r2,
        'lag_samples': lag, 'xcorr_score': xcorr_score, 'n_samples': int(trim.sum()),
    }


def summarize(df, label):
    return {
        'label': label,
        'n_trials': len(df),
        'rmse_mean': float(df['rmse_nmpkg'].mean()),
        'rmse_sd': float(df['rmse_nmpkg'].std(ddof=1)) if len(df) > 1 else 0.0,
        'r2_mean': float(df['r2_nmpkg'].mean()),
        'r2_sd': float(df['r2_nmpkg'].std(ddof=1)) if len(df) > 1 else 0.0,
    }

In [ ]:
manifest = list_awinda_trials()
hip_exo_df, knee_exo_df = load_exo_assisted_metrics()

hip_awinda_manifest = manifest[manifest['task'].isin(HIP_SHARED_TASKS)].copy()
knee_awinda_manifest = manifest[manifest['task'].isin(KNEE_SHARED_TASKS)].copy()

print(f'Awinda trials total: {len(manifest)}')
print(f'  hip shared (LG+RA): {len(hip_awinda_manifest)}')
print(f'  knee shared (RA+RD): {len(knee_awinda_manifest)}')
print(f'Exo assisted hip trials: {len(hip_exo_df)} | knee: {len(knee_exo_df)}')

hip_model, hip_window, _ = load_tcn(HIP_CKPT)
knee_model, knee_window, _ = load_tcn(KNEE_CKPT)

hip_awinda_rows = [process_awinda_hip_trial(r, hip_model, hip_window)
                   for _, r in hip_awinda_manifest.iterrows()]
knee_awinda_rows = [process_awinda_knee_trial(r, knee_model, knee_window)
                    for _, r in knee_awinda_manifest.iterrows()]

hip_awinda_df = pd.DataFrame(hip_awinda_rows)
knee_awinda_df = pd.DataFrame(knee_awinda_rows)

hip_summary = summarize(hip_exo_df, 'hip_exo_assisted')
hip_awinda_summary = summarize(hip_awinda_df, 'awinda_hip_only_exo_model')
knee_summary = summarize(knee_exo_df, 'knee_exo_assisted')
knee_awinda_summary = summarize(knee_awinda_df, 'awinda_knee_only_exo_model')

all_metrics = pd.concat([
    hip_exo_df.assign(source='assisted'),
    hip_awinda_df.assign(source='unassisted'),
    knee_exo_df.assign(source='assisted'),
    knee_awinda_df.assign(source='unassisted'),
], ignore_index=True)

summary_df = pd.DataFrame([hip_summary, hip_awinda_summary, knee_summary, knee_awinda_summary])

all_metrics.to_csv(CACHE_DIR / 'compare_assisted_unassisted_exo_models_trials.csv', index=False)
summary_df.to_csv(CACHE_DIR / 'compare_assisted_unassisted_exo_models_summary.csv', index=False)
summary_df.to_csv(OUT_DIR / 'table_overall.csv', index=False)

print('\n=== Paper bracket values (mean ± SD across trials) ===')
print(f"HIP EXO RMSE: {hip_summary['rmse_mean']:.3f} Nm/kg | R²: {hip_summary['r2_mean']:.3f}")
print(f"HIP-ONLY IMU RMSE: {hip_awinda_summary['rmse_mean']:.3f} Nm/kg | R²: {hip_awinda_summary['r2_mean']:.3f}")
print(f"KNEE EXO RMSE: {knee_summary['rmse_mean']:.3f} Nm/kg | R²: {knee_summary['r2_mean']:.3f}")
print(f"KNEE-ONLY IMU RMSE: {knee_awinda_summary['rmse_mean']:.3f} Nm/kg | R²: {knee_awinda_summary['r2_mean']:.3f}")

display(summary_df)
display(all_metrics.groupby(['platform', 'task'])[['rmse_nmpkg', 'r2_nmpkg']].agg(['mean', 'std', 'count']))

## Results interpretation

- **Hip**: RMSE is similar between assisted exo and unassisted Awinda hip-only replay through the same checkpoint; R² is moderately lower unassisted (0.72 vs 0.79), consistent with skin-mounted IMU IK vs device encoders.
- **Knee**: The finetuned knee-exo checkpoint (`0707_…`, no-LPF encoder pipeline) does **not** transfer to Awinda IMU IK kinematics (negative pooled R²). The model was trained/finetuned on exoskeleton encoder + gyro velocity streams; Awinda provides skin-mounted IK angles only, so this gap is expected and supports the paper’s caveat about hardware / pipeline differences.
- **Outputs**: `analysis/cache/compare_assisted_unassisted_exo_models_{trials,summary}.csv`, `analysis/paper_outputs/assisted_unassisted/`.

In [ ]:
latex = f"""\\subsection{{Estimation During Assisted and Unassisted Walking — numeric values}}

Across the shared hip conditions (LG + RA), the assisted hip-exoskeleton trials produced a mean RMSE of {hip_summary['rmse_mean']:.3f}~Nm/kg and a mean $R^2$ of {hip_summary['r2_mean']:.3f}, compared with {hip_awinda_summary['rmse_mean']:.3f}~Nm/kg and {hip_awinda_summary['r2_mean']:.3f} during the unassisted IMU trials (hip-only kinematics through the hip-exo TCN; $n={hip_awinda_summary['n_trials']}$).

Across the shared knee conditions (RA + RD), the assisted knee-exoskeleton trials produced a mean RMSE of {knee_summary['rmse_mean']:.3f}~Nm/kg and a mean $R^2$ of {knee_summary['r2_mean']:.3f}, compared with {knee_awinda_summary['rmse_mean']:.3f}~Nm/kg and {knee_awinda_summary['r2_mean']:.3f} for the unassisted knee-only IMU model ($n={knee_awinda_summary['n_trials']}$).
"""
tex_path = OUT_DIR / 'paper_snippet.tex'
tex_path.write_text(latex)
print(f'Wrote {tex_path}')
print(latex)